# CloudTrend P1–P3 R3 재현

2026-11-08 연구. 운영 DB·원장·스크리너에 접근하지 않는 오프라인 재현입니다. Drive R3 폴더의 ZIP과 SHA 파일을 사용하며, 모든 중간 결과와 재개 체크포인트를 새 Drive 폴더에 저장합니다. 사용된 과거 구간은 독립 OOS가 아닙니다.

기본 파일명: `CloudTrend_P123_R3_20261108.zip`. 직접 경로를 지정할 수 있습니다. 같은 `RUN_ID`로 재개하며, 코드/입력 변경 시 새 RUN_ID를 사용하세요.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
from pathlib import Path
import hashlib,json,subprocess,sys,zipfile,shutil,os
BUNDLE_PATH = ""  # 비워두면 CloudTrend 안에서 탐색
RUN_ID = "R3_reproduction_20261108"
root=Path("/content/drive/MyDrive/CloudTrend")
if BUNDLE_PATH:
    bundle=Path(BUNDLE_PATH)
else:
    matches=sorted(root.rglob("CloudTrend_P123_R3_20261108.zip"))
    if not matches: raise FileNotFoundError("CloudTrend R3 폴더의 ZIP 경로를 BUNDLE_PATH에 넣어주세요.")
    bundle=next((p for p in matches if "한국" in str(p)),matches[0])
print("재현 원본:",bundle)
sha_file=bundle.with_suffix(".sha256")
expected=sha_file.read_text().split()[0]
actual=hashlib.sha256(bundle.read_bytes()).hexdigest()
assert expected==actual,"ZIP SHA 불일치"
base=bundle.parent/"R3_reproduction_workspace"
workspace=base/"CloudTrend_P123_R3_20261108"
if not (workspace/"DELIVERY_MANIFEST.json").exists():
    base.mkdir(parents=True,exist_ok=True)
    with zipfile.ZipFile(bundle) as z:
        for n in z.namelist():
            if not (base/n).resolve().is_relative_to(base.resolve()): raise ValueError("Unsafe archive member")
        z.extractall(base)
manifest=json.loads((workspace/"DELIVERY_MANIFEST.json").read_text())
for f in manifest["files"]:
    assert hashlib.sha256((workspace/f["path"]).read_bytes()).hexdigest()==f["sha256"],f["path"]
print("전체 전달 파일 SHA 검증 완료:",len(manifest["files"]))
run_dir=bundle.parent/"재현실행"/RUN_ID
run_dir.mkdir(parents=True,exist_ok=True)
print("자동 저장/재개 위치:",run_dir)


In [ ]:
subprocess.run([sys.executable,"-m","pip","install","-q","numpy==2.3.5","pandas==2.2.3","pyarrow==26.0.1"],check=True)
node=shutil.which("node")
major=int(subprocess.check_output([node,"--version"],text=True).strip().lstrip("v").split(".")[0]) if node else 0
if major<24:
    subprocess.run(["npm","install","--prefix","/content/cloudtrend-node24","node@24.22.1"],check=True)
    node="/content/cloudtrend-node24/node_modules/node/bin/node"
print("Node:",subprocess.check_output([node,"--version"],text=True).strip())


In [ ]:
cmd=[sys.executable,str(workspace/"cloudtrend/research/p123_r3/run_all.py"),"--workspace",str(workspace),"--run-dir",str(run_dir),"--node",node]
proc=subprocess.Popen(cmd,cwd=workspace,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
with (run_dir/"colab_console.log").open("a") as log:
    for line in proc.stdout:
        print(line,end="",flush=True);log.write(line);log.flush()
rc=proc.wait()
if rc: raise RuntimeError("재현 중단: Drive 로그를 확인한 뒤 같은 RUN_ID로 재개하세요.")
print(json.loads((run_dir/"COMPLETE.json").read_text()))


최종 표는 `recent/comparison.csv`, `long-analysis/kr_portfolio_comparison.csv`, `official-analysis/official_account_comparison.csv`, `diagnostics/signal_availability_sensitivity.csv`입니다. 장기 미국 표는 기존 완료 진입 선택 비교이며 새 장기 재배정 계좌가 아닙니다. 한국 11/6 전략 신호 미완성과 과거 종목군 PIT 한계는 보고서에 명시되어 있습니다. 자동매매나 자동 운영 적용은 수행하지 않습니다.